In [1]:
print("Hello")

Hello


In [47]:
# here i have use (nmap -sV -oX investigation.xml scanme.nmap.org) to scan the target and save the output in XML format in the same directory of this file
#  i  have used -sV  Service Version Detection and -oX to save the output in XML format

In [ ]:
import xml.etree.ElementTree as ET
import requests
import os # i used it because .env is external file so thta i have used it here 
from dotenv import load_dotenv  # here Why I iam using Like to hide my Api Key To everyone  iam importing this library 
load_dotenv()
API_KEY = os.getenv("VT_API_KEY") # here I am getting my API Key from .env file  that i nahve been created already
headers = {"x-apikey": API_KEY}


In [5]:
try:
    tree = ET.parse("investigation.xml")
    root = tree.getroot()
except Exception as e:
    print("Error loading XML:", e)
    exit() # Exit if XML loading fails here 

In [8]:
for host in root.findall("host"):

    address = host.find("address")
    if address is None:
        continue

    ip = address.attrib.get("addr")
print("\n")
print("Host IP:", ip)
print(f"{'-'*40}")



Host IP: 45.33.32.156
----------------------------------------


In [55]:
#here is the virustotal url  in that i used ip to scan that ip address  and get the information abot that ip address
vt_url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
try:
        vt_response = requests.get(vt_url, headers=headers)

        if vt_response.status_code == 200:
            vt_data = vt_response.json()
            stats = vt_data["data"]["attributes"]["last_analysis_stats"]

            print("\n[ VirusTotal Analysis ]")
            print("Malicious :", stats.get("malicious", 0))
            print("Suspicious:", stats.get("suspicious", 0))
            print("Harmless  :", stats.get("harmless",0))

        else:
            print("\nVirusTotal Error:", vt_response.status_code)

except Exception as e:
        print("VirusTotal Request Failed:", e)
        # here iam using try and except if virustotal doesnot work it through an error so that i can reslove the error easily


[ VirusTotal Analysis ]
Malicious : 4
Suspicious: 1
Harmless  : 55


In [ ]:
# iam checking Vendor of malicious 
malicious_vendors = {
    vendor: details 
    for vendor, details in vt_data["data"]["attributes"]["last_analysis_results"].items() 
    if details.get("category") == "malicious"
}

# Extract just the vendor names
malicious_engine_names = list(malicious_vendors.keys())

print("Malicious Vendors/Engines:")
for vendor in malicious_engine_names:
    print(f"  - {vendor}")

Malicious Vendors/Engines:
  - ADMINUSLabs
  - Criminal IP
  - CyRadar
  - ESET


In [ ]:
# for suspicious vendor
suspicious_vendors = {
    vendor: details 
    for vendor, details in vt_data["data"]["attributes"]["last_analysis_results"].items() 
    if details.get("category") == "suspicious"
}
suspicious_engine_names = list(suspicious_vendors.keys())
print("\nSuspicious Vendors/Engines:")
for vendor in suspicious_engine_names:
    print(f"  - {vendor}")


Suspicious Vendors/Engines:
  - SOCRadar


In [54]:
#for ports and Services
print("\nOpen Ports and Services:")

ports = host.find("ports")

for port in ports.findall("port"):
    state = port.find("state")
    if state is None or state.attrib.get("state") != "open":
        continue
    port_id = port.attrib.get("portid")
    protocol = port.attrib.get("protocol")

    service = port.find("service")
    service_name = service.attrib.get("name", "unknown")
    product = service.attrib.get("product", "")
    version = service.attrib.get("version", "")
    extra = service.attrib.get("extrainfo", "")

    print("\nPort:", f"{port_id}/{protocol}")
    print("Service :", service_name)
    print("Product :", product)
    print("Version :", version)
    print("Extra   :", extra)


Open Ports and Services:

Port: 22/tcp
Service : ssh
Product : OpenSSH
Version : 6.6.1p1 Ubuntu 2ubuntu2.13
Extra   : Ubuntu Linux; protocol 2.0

Port: 80/tcp
Service : http
Product : Apache httpd
Version : 2.4.7
Extra   : (Ubuntu)

Port: 9929/tcp
Service : nping-echo
Product : Nping echo
Version : 
Extra   : 

Port: 31337/tcp
Service : tcpwrapped
Product : 
Version : 
Extra   : 


In [ ]:
stats = vt_data["data"]["attributes"].get("last_analysis_stats", {})
print("\n[ Reputation / Detection Results ]")
print("Malicious :", stats.get("malicious", 0))
print("Suspicious:", stats.get("suspicious", 0))
print("Harmless  :", stats.get("harmless", 0))
# Domain & IP Intelligence
print("\n[ Domain & IP Intelligence ]")
print("Registrar :",stats.get("registrar"))
print("Creation Date :",stats.get("creation_date"))
print("Last DNS Records :",stats.get("last_dns_records", [])[:3])

 # Historical / Contextual data
print("\n[ Historical / Contextual Data ]")
print("Last Analysis Date :",stats.get("last_analysis_date"))
print("Popularity Ranks :",stats.get("popularity_ranks"))

# Security observations
print("\n[ Security Observations ]")
if stats.get("malicious", 0) > 0:
    print(" Malicious detections found.")
elif stats.get("suspicious", 0) > 0:
    print(" Suspicious activity detected.")
else:
    print("✔ No major detections reported.")

"""
Questions:
1.What VirusTotal is used for?
Ans: VirusTotal is a threat intelligence platform used to analyze
(The Domains,IP addresses,Files,URLs)
by that  we can analyze the data and get the information about  that and also  is there any malicious activity is there
or not we can find it and we can take the action according to that information
and also we can take precautions to protect our system from that malicious activity

2.How passive intelligence differs from active scanning ?
Passive Intelligence is used on the exicting files 
and also  so  that No traffic will be generated because we have file already
example: VirusTotal, Hybrid Analysis, etc.
Active Scanning is used to scan the target and get the information about that target
and also  so that Traffic will be generated because we are scanning the target
Example: Nmap

3.How VirusTotal findings support or complement your Nmap results?

Nmap and VirusTotal provide different but complementary perspectives.
Nmap Provides:  Open ports,Running services,Service versions,Host exposure details
VirusTotal Provides:Reputation of domain/IP,Security detections,Historical context,Threat intelligence data
By combining both:
Nmap explains what is running on a system.
VirusTotal explains how the system is viewed from a security perspective.
This allows analysts to perform better risk assessment without exploitation.
both technical and security context, improving overall analysis and 
helping identify possible risks associated with exposed services.

"""




[ Reputation / Detection Results ]
Malicious : 4
Suspicious: 1
Harmless  : 55

[ Domain & IP Intelligence ]
Registrar : None
Creation Date : None
Last DNS Records : []

[ Historical / Contextual Data ]
Last Analysis Date : None
Popularity Ranks : None

[ Security Observations ]
 Malicious detections found.


'\nQuestions:\n1.What VirusTotal is used for?\nAns: VirusTotal is a threat intelligence platform used to analyze\n(The Domains,IP addresses,Files,URLs)\nby that  we can analyze the data and get the information about  that and also  is there any malicious activity is there\nor not we can find it and we can take the action according to that information\nand also we can take precautions to protect our system from that malicious activity\n\n2.How passive intelligence differs from active scanning ?\nPassive Intelligence is used on the exicting files \nand also  so  that No traffic will be generated because we have file already\nexample: VirusTotal, Hybrid Analysis, etc.\nActive Scanning is used to scan the target and get the information about that target\nand also  so that Traffic will be generated because we are scanning the target\nExample: Nmap\n\n'

In [64]:
#Task 5

"""
1.
Nmap Commands Used
{ [ nmap]  is a network scanning tool
  [scanme.nmap.org ] is target host  }

here i have used [ -sV ]  so that i can detects running services and versions 
there also many Commands like 
sudo nmap -A scanme.nmap.org (this command that used for agressive scan )
nmap -p- scanme.nmap.org (this command that used for scan all ports)


2. Scan Results

Open Ports and Services:

Port: 22/tcp
Service : ssh
Product : OpenSSH
Version : 6.6.1p1 Ubuntu 2ubuntu2.13
Extra   : Ubuntu Linux; protocol 2.0

Port: 80/tcp
Service : http
Product : Apache httpd
Version : 2.4.7
Extra   : (Ubuntu)

Port: 9929/tcp
Service : nping-echo
Product : Nping echo
Version : 
Extra   : 

Port: 31337/tcp
Service : tcpwrapped
Product : 
Version : 
Extra   : 


3. VirusTotal Findings
Domain reputation information collected
Detection statistics (malicious / suspicious / harmless)
Domain and IP intelligence data
Historical analysis information

here i found  like some malicious activity in my system 
then i have checked what are they how  many and its names 
Malicious : 4
Suspicious: 1
Harmless  : 55

Malicious names are:
Malicious Vendors/Engines:
  - ADMINUSLabs
  - Criminal IP
  - CyRadar
  - ESET

And also  i found  Suspicious things  also :
Suspicious Vendors/Engines:
  - SOCRadar


4 .Port & Service Analysis
Open Ports and Services:

Port: 22/tcp
Service : ssh
Product : OpenSSH
Version : 6.6.1p1 Ubuntu 2ubuntu2.13
Extra   : Ubuntu Linux; protocol 2.0

Port: 80/tcp
Service : http
Product : Apache httpd
Version : 2.4.7
Extra   : (Ubuntu)

Port: 9929/tcp
Service : nping-echo
Product : Nping echo
Version : 
Extra   : 

Port: 31337/tcp
Service : tcpwrapped
Product : 
Version : 
Extra   : 

5. Vulnerability Reasoning
Older software versions may contain known vulnerabilities.
SSH service could be targeted by unauthorized login attempts.
Web server may have security risks.
Additional open ports increase attack surface
No exploitation performed.

This exercise helped me understand: 
- How to use Nmap for network scanning and service detection.
- How to analyze scan results and identify open ports and services.
- How to use VirusTotal for threat intelligence and reputation analysis.
- The importance of combining technical and security context for better analysis.
- and  while scaning my traget i found malicious  first   i thought  that maliciouos is there in my system 
  then i  have checked that malicious names  
   They are used to detect threats, not to harm our computer Systems 
   and they are trusted cybersecurity vendors, so they are not dangerous to your system. 
   by this activity i know  a Good Information 
   about active scanning and passive intelligence 
   Combining Nmap and VirusTotal provides better understanding of system exposure and security risks



"""

'\n1.\nNmap Commands Used\n{ [ nmap]  is a network scanning tool\n  [scanme.nmap.org ] is target host  }\n\nhere i have used [ -sV ]  so that i can detects running services and versions \nthere also many Commands like \nsudo nmap -A scanme.nmap.org (this command that used for agressive scan )\nnmap -p- scanme.nmap.org (this command that used for scan all ports)\n\n\n2. Scan Results\n\nOpen Ports and Services:\n\nPort: 22/tcp\nService : ssh\nProduct : OpenSSH\nVersion : 6.6.1p1 Ubuntu 2ubuntu2.13\nExtra   : Ubuntu Linux; protocol 2.0\n\nPort: 80/tcp\nService : http\nProduct : Apache httpd\nVersion : 2.4.7\nExtra   : (Ubuntu)\n\nPort: 9929/tcp\nService : nping-echo\nProduct : Nping echo\nVersion : \nExtra   : \n\nPort: 31337/tcp\nService : tcpwrapped\nProduct : \nVersion : \nExtra   : \n\n\n3. VirusTotal Findings\nDomain reputation information collected\nDetection statistics (malicious / suspicious / harmless)\nDomain and IP intelligence data\nHistorical analysis information\n\nhere i fo